# Notebook 12 — Mini-Project and Assessment

**Module:** 15 — Simulation and Agent-Based Modeling  
**Tier:** 2 — Working competence  
**Notebook:** 12 of 12  
**Time estimate:** 4–6 hours (50 points)

> Two parts:
> - **MP01:** Epidemic model comparison — ODE SIR vs. network SIR vs. ABM SIR
> - **Assessment:** 5 × 10 points covering ODE analysis, stochastic simulation,
>   ABM reasoning, parameter estimation, and bifurcation.

---
## MP01 — Epidemic Model Comparison: Three Levels of Abstraction

**Goal:** compare the ODE SIR, the network SIR (on a Barabási-Albert graph),
and a spatial ABM SIR — all with the same β and γ. Identify when the simple
ODE approximation fails and what the richer models reveal.

**This is direct interview preparation.** The question "what's the difference
between a compartmental ODE model and an ABM for epidemic spread" appears
frequently in computational biology and epidemiology interviews.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.signal import convolve2d

rng = np.random.default_rng(42)

# ============================================================
# Shared epidemic parameters
# ============================================================
N_POP = 500
BETA  = 0.30   # per-contact transmission rate
GAMMA = 0.10   # recovery rate
I0    = 5      # initial infected
N_REPS = 20    # stochastic replicates

# ============================================================
# Model 1: ODE SIR
# ============================================================
def ode_sir(t, y, beta=BETA, gamma=GAMMA, N=N_POP):
    S, I, R = y
    return [-beta*S*I/N, beta*S*I/N - gamma*I, gamma*I]

T_ODE = np.linspace(0, 200, 1000)
sol_ode = solve_ivp(ode_sir, (0, 200), [N_POP-I0, I0, 0],
                      t_eval=T_ODE, method='RK45', max_step=0.5)
S_ode = sol_ode.y[0] / N_POP
I_ode = sol_ode.y[1] / N_POP
R_ode = sol_ode.y[2] / N_POP
print(f'ODE SIR: peak I={I_ode.max():.3f}, final R={R_ode[-1]:.3f}')

# ============================================================
# Model 2: Network SIR on BA graph
# ============================================================
try:
    import networkx as nx
    G_ba = nx.barabasi_albert_graph(N_POP, 3, seed=42)
    adj_net = {n: list(G_ba.neighbors(n)) for n in G_ba.nodes()}
except ImportError:
    # Fallback: random adjacency
    adj_net = {i: [] for i in range(N_POP)}
    for i in range(N_POP):
        for j in range(i+1, N_POP):
            if rng.random() < 6/N_POP:
                adj_net[i].append(j); adj_net[j].append(i)

def network_sir_run(adj, N, beta, gamma, I0, seed):
    rng_s = np.random.default_rng(seed)
    state = np.zeros(N, dtype=int)
    init_inf = rng_s.choice(N, I0, replace=False)
    state[init_inf] = 1
    S_t, I_t, R_t = [(state==0).sum()], [(state==1).sum()], [(state==2).sum()]
    for _ in range(300):
        if (state==1).sum() == 0:
            break
        new_state = state.copy()
        recover = (state==1) & (rng_s.random(N) < gamma)
        new_state[recover] = 2
        for node in range(N):
            if state[node] == 0:
                n_inf_neigh = sum(1 for nb in adj.get(node,[]) if state[nb]==1)
                if n_inf_neigh > 0 and rng_s.random() < 1-(1-beta)**n_inf_neigh:
                    new_state[node] = 1
        state = new_state
        S_t.append((state==0).sum()); I_t.append((state==1).sum()); R_t.append((state==2).sum())
    T_max = len(S_t)
    return (np.array(S_t)/N, np.array(I_t)/N, np.array(R_t)/N)

print(f'Running {N_REPS} network SIR replicates...')
net_runs = [network_sir_run(adj_net, N_POP, BETA, GAMMA, I0, seed=i+10) for i in range(N_REPS)]
net_peaks = [r[1].max() for r in net_runs]
net_finals = [r[2][-1] for r in net_runs]
print(f'Network SIR: mean peak I={np.mean(net_peaks):.3f}±{np.std(net_peaks):.3f}, mean final R={np.mean(net_finals):.3f}')

# ============================================================
# Model 3: Spatial ABM SIR
# ============================================================
def spatial_abm_sir(N_grid=30, N_agents=500, beta=BETA, gamma=GAMMA, I0=5, n_steps=300, seed=42):
    """
    Grid-based ABM SIR: agents move on a 2D grid.
    Transmission: susceptible agent infected if any infected agent
    in same or adjacent cell (Moore neighbourhood).
    """
    rng_a = np.random.default_rng(seed)
    pos   = rng_a.integers(0, N_grid, (N_agents, 2))
    state = np.zeros(N_agents, dtype=int)  # 0=S, 1=I, 2=R
    init_inf = rng_a.choice(N_agents, I0, replace=False)
    state[init_inf] = 1

    S_t, I_t, R_t = [(state==0).sum()], [(state==1).sum()], [(state==2).sum()]
    for _ in range(n_steps):
        if (state==1).sum() == 0:
            break
        # Build spatial occupancy grid of infected agents
        inf_grid = np.zeros((N_grid, N_grid), dtype=int)
        for i in range(N_agents):
            if state[i] == 1:
                inf_grid[pos[i,0], pos[i,1]] += 1
        # Neighbourhood infected count (Moore)
        neigh_inf = convolve2d(inf_grid, np.ones((3,3), dtype=int),
                                 mode='same', boundary='wrap')
        new_state = state.copy()
        for i in range(N_agents):
            r, c = pos[i]
            if state[i] == 0:
                n_inf = neigh_inf[r, c] - (1 if state[i]==1 else 0)
                if n_inf > 0 and rng_a.random() < 1-(1-beta)**n_inf:
                    new_state[i] = 1
            elif state[i] == 1:
                if rng_a.random() < gamma:
                    new_state[i] = 2
        state = new_state
        # Random movement
        moves = rng_a.integers(-1, 2, (N_agents, 2))
        pos = (pos + moves) % N_grid
        S_t.append((state==0).sum()); I_t.append((state==1).sum()); R_t.append((state==2).sum())
    return np.array(S_t)/N_agents, np.array(I_t)/N_agents, np.array(R_t)/N_agents

print(f'Running {N_REPS} spatial ABM SIR replicates...')
abm_runs = [spatial_abm_sir(seed=i+100) for i in range(N_REPS)]
abm_peaks  = [r[1].max() for r in abm_runs]
abm_finals = [r[2][-1] for r in abm_runs]
print(f'ABM SIR: mean peak I={np.mean(abm_peaks):.3f}±{np.std(abm_peaks):.3f}, mean final R={np.mean(abm_finals):.3f}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# Panel A: ODE SIR
ax = axes[0, 0]
ax.plot(T_ODE, S_ode, 'steelblue', lw=2, label='S')
ax.plot(T_ODE, I_ode, 'tomato', lw=2, label='I')
ax.plot(T_ODE, R_ode, 'green', lw=2, label='R')
ax.set_xlabel('Days'); ax.set_ylabel('Fraction')
ax.set_title(f'A. ODE SIR (deterministic)\nR₀={BETA/GAMMA:.1f}, final R={R_ode[-1]:.3f}')
ax.legend(fontsize=9)

# Panel B: Network SIR ensemble
ax = axes[0, 1]
for S, I, R in net_runs:
    ax.plot(range(len(I)), I, 'tomato', lw=0.5, alpha=0.3)
mean_I_net = np.zeros(200)
for S, I, R in net_runs:
    n = min(len(I), 200)
    mean_I_net[:n] += I[:n] / N_REPS
ax.plot(range(200), mean_I_net, 'tomato', lw=2.5, label='Mean I(t)')
ax.plot(T_ODE[:200], I_ode[:200], 'k--', lw=2, label='ODE reference')
ax.set_xlabel('Step'); ax.set_ylabel('Fraction infectious')
ax.set_title(f'B. Network SIR ({N_REPS} replicates)\nvs. ODE (dashed)')
ax.legend(fontsize=8)

# Panel C: Spatial ABM ensemble
ax = axes[0, 2]
for S, I, R in abm_runs:
    ax.plot(range(len(I)), I, 'steelblue', lw=0.5, alpha=0.3)
mean_I_abm = np.zeros(200)
for S, I, R in abm_runs:
    n = min(len(I), 200)
    mean_I_abm[:n] += I[:n] / N_REPS
ax.plot(range(200), mean_I_abm, 'steelblue', lw=2.5, label='Mean I(t)')
ax.plot(T_ODE[:200], I_ode[:200], 'k--', lw=2, label='ODE reference')
ax.set_xlabel('Step'); ax.set_ylabel('Fraction infectious')
ax.set_title(f'C. Spatial ABM SIR ({N_REPS} replicates)\nvs. ODE (dashed)')
ax.legend(fontsize=8)

# Panel D: Peak infected comparison (violin)
ax = axes[1, 0]
plot_data = [net_peaks, abm_peaks]
parts = ax.violinplot(plot_data, positions=[1, 2], showmeans=True)
for pc in parts['bodies']:
    pc.set_alpha(0.7)
ax.axhline(I_ode.max(), color='k', ls='--', lw=1.5, label=f'ODE peak={I_ode.max():.3f}')
ax.set_xticks([1, 2]); ax.set_xticklabels(['Network SIR', 'Spatial ABM'])
ax.set_ylabel('Peak infected fraction')
ax.set_title('D. Peak infection comparison\n(violin plots, 20 replicates each)')
ax.legend(fontsize=9)

# Panel E: Final epidemic size comparison
ax = axes[1, 1]
ax.scatter([1]*N_REPS, net_finals,  alpha=0.7, s=30, color='tomato',    label='Network SIR')
ax.scatter([2]*N_REPS, abm_finals,  alpha=0.7, s=30, color='steelblue', label='Spatial ABM')
ax.scatter([3], [R_ode[-1]], s=100, color='black', marker='D', label='ODE SIR', zorder=5)
ax.errorbar([1], np.mean(net_finals), yerr=np.std(net_finals), color='tomato', capsize=5, lw=2)
ax.errorbar([2], np.mean(abm_finals), yerr=np.std(abm_finals), color='steelblue', capsize=5, lw=2)
ax.set_xticks([1,2,3]); ax.set_xticklabels(['Network\nSIR', 'Spatial\nABM', 'ODE\nSIR'])
ax.set_ylabel('Final epidemic size (R∞)')
ax.set_title('E. Final epidemic size\n(ODE vs. stochastic models)')
ax.legend(fontsize=8)

# Panel F: Model comparison summary table
ax = axes[1, 2]
ax.axis('off')
rows = [
    ['Property', 'ODE SIR', 'Network SIR', 'Spatial ABM'],
    ['Type', 'Deterministic', 'Stochastic', 'Stochastic'],
    ['Mixing', 'Homogeneous', 'Heterogeneous', 'Local (spatial)'],
    ['Stochasticity', 'None', 'Contact network', 'Space + contacts'],
    ['Individual variation', 'None', 'Degree variation', 'Position variation'],
    ['Peak I', f'{I_ode.max():.3f}', f'{np.mean(net_peaks):.3f}±{np.std(net_peaks):.3f}',
       f'{np.mean(abm_peaks):.3f}±{np.std(abm_peaks):.3f}'],
    ['Final R∞', f'{R_ode[-1]:.3f}', f'{np.mean(net_finals):.3f}±{np.std(net_finals):.3f}',
       f'{np.mean(abm_finals):.3f}±{np.std(abm_finals):.3f}'],
    ['Appropriate when', 'N→∞, random mixing', 'Network structure matters', 'Spatial clustering matters'],
]
table = ax.table(cellText=rows[1:], colLabels=rows[0], loc='center', cellLoc='center')
table.auto_set_font_size(False); table.set_fontsize(7.5)
table.scale(1, 1.9)
ax.set_title('F. Three SIR models: comparison')

plt.suptitle('Module 15 NB12 MP01: Epidemic Model Comparison\n(ODE vs. Network vs. Spatial ABM SIR)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('mini_project_mp01_epidemic.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Assessment (50 points)

**Instructions:**
- Write all answers in `exercises/12_assessment_answers.md`
- Each question is 10 points (partial credit given)
- Show all mathematical derivations step by step

---

### Q1 — SIR ODE analysis (10 points)

The SIR model:
$\dot{S} = -\beta SI/N$, $\dot{I} = \beta SI/N - \gamma I$, $\dot{R} = \gamma I$.

**(a)** Derive the condition dI/dt > 0 (epidemic grows). Express in terms of R₀ and S/N.

**(b)** For measles (R₀ = 15), what fraction of the population must be immune (from
vaccination or past infection) to prevent a new epidemic? Show the derivation.

**(c)** The final epidemic size r∞ satisfies the equation $r_\infty = 1 - e^{-R_0 r_\infty}$.
Solve this numerically for R₀ = 2.0, 3.0, 5.0. Show the Newton-Raphson iteration
for at least one value.

---

### Q2 — Gillespie SSA derivation (10 points)

Consider a simple system with two reactions:
- Reaction 1: $A \to B$ with rate constant $c_1 = 0.1$ (propensity $a_1 = c_1 N_A$)
- Reaction 2: $B \to A$ with rate constant $c_2 = 0.05$ (propensity $a_2 = c_2 N_B$)

Starting from $N_A = 100$, $N_B = 0$.

**(a)** Write the propensities $a_1$ and $a_2$ at the initial state. What is the
total propensity $a_0$?

**(b)** Draw $u_1 = 0.37$ and $u_2 = 0.85$ from $U(0,1)$. Compute the time to the
next reaction $\tau$ and determine which reaction fires.

**(c)** What is the equilibrium mean of $N_A$? Derive it from detailed balance
($a_1 = a_2$ at steady state). How does the Gillespie SSA approach this equilibrium?

---

### Q3 — ABM vs. ODE (10 points)

**(a)** The logistic growth ODE $dN/dt = rN(1-N/K)$ predicts a smooth S-shaped
growth curve to carrying capacity K. Describe two biological phenomena that this
ODE cannot capture but an ABM could. Explain why.

**(b)** In the Schelling segregation model, increasing the satisfaction threshold
from 40% to 60% increases segregation. Show that this is counterintuitive (more
tolerant individuals produce more segregation) and explain the mechanism.

**(c)** Implement a Boids-style flocking model from scratch (10 lines of pseudocode):
agents follow three rules — separation (avoid crowding), alignment (match
neighbours' velocity), cohesion (move toward the group centre). Describe what
collective behaviour emerges and why it cannot be derived analytically from the rules.

---

### Q4 — Parameter estimation (10 points)

**(a)** You fit an SIR model to 30 days of early epidemic data and get β=0.35±0.12
and γ=0.12±0.08. A colleague fits the same data and gets R₀=2.9±0.3. Explain
why the R₀ estimate is more precise than the individual β and γ estimates even
though R₀ = β/γ.

**(b)** The profile likelihood for γ has a flat bottom (almost no increase in NLL
for γ ∈ [0.05, 0.25]). What does this mean for the identifiability of γ? What
additional data would make γ identifiable?

**(c)** Describe the Metropolis-Hastings acceptance criterion. A proposed parameter
θ' has log-likelihood -150; the current parameter θ has log-likelihood -145.
What is the acceptance probability? When would you accept a proposal with lower
likelihood?

---

### Q5 — Bifurcation and bistability (10 points)

The toggle switch ODE:
$\dot{u} = \alpha/(1+v^n) - u$, $\dot{v} = \alpha/(1+u^n) - v$

**(a)** Find the symmetric fixed point ($u^* = v^*$) algebraically. For $n=2$, $\alpha=4$:
solve for $u^*$ and classify the fixed point's stability.

**(b)** Draw a schematic bifurcation diagram for this system varying $\alpha$ from 1 to 8,
showing both stable branches and the unstable branch. Label the saddle-node
bifurcation points and the bistable region.

**(c)** A biologist proposes using the toggle switch as a cell fate switch by
adding a transient signal that pushes the cell from the low-u/high-v state to
the high-u/low-v state. The signal is applied for 2 hours. Describe the minimum
signal strength required for a permanent switch (in terms of the phase portrait)
and why no permanent switch occurs if the signal is too weak.

---

In [ ]:
# Assessment stubs — implement these in exercises/12_assessment_answers.md
# and in this cell. All raise NotImplementedError until you implement them.

def sir_final_size(R0):
    """
    Q1(c): Find the final epidemic size r_inf satisfying r_inf = 1 - exp(-R0 * r_inf).
    Returns: float in [0, 1]
    """
    raise NotImplementedError('Q1: implement SIR final size equation solver')

def gillespie_one_step(NA, NB, c1, c2, u1, u2):
    """
    Q2(b): Given the two uniform random numbers u1 and u2,
    compute tau (time to next reaction) and which reaction fires (0 or 1).
    Returns: (tau, reaction_index)
    """
    raise NotImplementedError('Q2: implement one Gillespie step')

def acceptance_probability(log_lik_proposed, log_lik_current):
    """
    Q4(c): Metropolis acceptance probability.
    Returns: float in [0, 1]
    """
    raise NotImplementedError('Q4: implement Metropolis acceptance probability')

def toggle_symmetric_fixed_point(alpha, n):
    """
    Q5(a): Find the symmetric fixed point u* = v* of the toggle switch.
    Solve u* = alpha / (1 + u*^n) numerically.
    Returns: float (the fixed point u*)
    """
    raise NotImplementedError('Q5: implement toggle switch fixed point finder')


# ---- Evaluation harness ----
import traceback

def test_sir_final():
    try:
        r_inf = sir_final_size(2.0)
        assert 0.7 < r_inf < 0.9, f'R0=2: expected ~0.797, got {r_inf:.3f}'
        r_inf3 = sir_final_size(3.0)
        assert 0.90 < r_inf3 < 0.97, f'R0=3: expected ~0.940, got {r_inf3:.3f}'
        print('[Q1] sir_final_size: PASS')
    except NotImplementedError:
        print('[Q1] sir_final_size: NOT IMPLEMENTED')
    except Exception as e:
        print(f'[Q1] sir_final_size: ERROR — {e}')

def test_gillespie_step():
    try:
        tau, rxn = gillespie_one_step(100, 0, 0.1, 0.05, 0.37, 0.85)
        # a0 = c1*100 = 10; tau = -ln(0.37)/10 ≈ 0.0994
        assert abs(tau - (-np.log(0.37)/10)) < 0.001, f'tau wrong: {tau}'
        # u2*a0 = 0.85*10=8.5 > a1=10, so reaction 2 fires
        # Actually a1=10, a0=10, so u2*a0=8.5 < a1=10, reaction 0 fires
        assert rxn == 0, f'Expected reaction 0, got {rxn}'
        print('[Q2] gillespie_one_step: PASS')
    except NotImplementedError:
        print('[Q2] gillespie_one_step: NOT IMPLEMENTED')
    except Exception as e:
        print(f'[Q2] gillespie_one_step: ERROR — {e}')

def test_acceptance():
    try:
        p = acceptance_probability(-150, -145)
        expected = np.exp(-150 - (-145))
        assert abs(p - expected) < 1e-6, f'Expected {expected:.4f}, got {p:.4f}'
        p_better = acceptance_probability(-140, -145)
        assert p_better == 1.0, 'Better proposal should always be accepted'
        print('[Q4] acceptance_probability: PASS')
    except NotImplementedError:
        print('[Q4] acceptance_probability: NOT IMPLEMENTED')
    except Exception as e:
        print(f'[Q4] acceptance_probability: ERROR — {e}')

def test_toggle_fp():
    try:
        u_star = toggle_symmetric_fixed_point(alpha=4.0, n=2)
        # u* satisfies u* = 4/(1+u*^2)
        residual = abs(u_star - 4.0/(1+u_star**2))
        assert residual < 1e-5, f'Residual {residual:.6f} too large'
        assert 0.8 < u_star < 1.2, f'Expected u*~1.0 for alpha=4,n=2, got {u_star:.4f}'
        print('[Q5] toggle_symmetric_fixed_point: PASS')
    except NotImplementedError:
        print('[Q5] toggle_symmetric_fixed_point: NOT IMPLEMENTED')
    except Exception as e:
        print(f'[Q5] toggle_symmetric_fixed_point: ERROR — {e}')

print('=== Module 15 Assessment Evaluation ===')
test_sir_final()
test_gillespie_step()
test_acceptance()
test_toggle_fp()


<!-- REFERENCE IMPLEMENTATIONS (do not read before attempting)

def sir_final_size(R0):
    from scipy.optimize import brentq
    if R0 <= 1: return 0.0
    return brentq(lambda r: r - (1 - np.exp(-R0*r)), 1e-9, 1-1e-9)

def gillespie_one_step(NA, NB, c1, c2, u1, u2):
    a1 = c1 * NA
    a2 = c2 * NB
    a0 = a1 + a2
    tau = -np.log(u1) / a0
    rxn = 0 if u2 * a0 < a1 else 1
    return tau, rxn

def acceptance_probability(log_lik_proposed, log_lik_current):
    return min(1.0, np.exp(log_lik_proposed - log_lik_current))

def toggle_symmetric_fixed_point(alpha, n):
    from scipy.optimize import brentq
    # u* = alpha/(1 + u*^n)  =>  u*(1+u*^n) - alpha = 0
    return brentq(lambda u: u*(1+u**n) - alpha, 1e-9, alpha)

-->

---
## Module 15 — Completion Checklist

- [ ] NB01: Can implement RK4 and SIR from scratch, derive herd immunity threshold
- [ ] NB02: Can implement Gillespie SSA from propensity functions
- [ ] NB03: Can implement Gray-Scott using finite differences
- [ ] NB04: Can implement Game of Life and explain excitable medium CAs
- [ ] NB05: Can build an ABM with Agent/Model/Scheduler from scratch
- [ ] NB06: Can implement Wright-Fisher model and explain genetic drift
- [ ] NB07: Can run SIR on a BA network and explain the epidemic threshold
- [ ] NB08: Can implement random walk, chemotaxis, and compute MSD
- [ ] NB09: Can fit ODE parameters with MCMC and interpret the posterior
- [ ] NB10: Can draw a bifurcation diagram and explain bistability
- [ ] NB11: Can implement a hybrid ODE + ABM tumor growth model
- [ ] NB12 (this): MP01 complete, all 5 assessment questions answered
- [ ] All exercises in `exercises/` attempted before reading reference implementations

---
*Module 15 complete. Proceed to Module 16: Research Software Engineering.*